# 09 人工审批与客服接管

**用途：** 分别验证高风险工具审批和AI无法处理时的人工客服回复恢复。

> 使用方式：按顺序运行。出现 `PASS` 才代表本节验收成功；断言失败时先阅读紧邻的“失败定位”。默认不调用真实模型、不写生产数据库。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys
import tempfile

cwd = Path.cwd().resolve()
DAY1_ROOT = None
PROJECT2_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "project2" / "agent_graph.py").exists():
        DAY1_ROOT = candidate
        PROJECT2_ROOT = candidate / "project2"
        break
    if (candidate / "agent_graph.py").exists() and (candidate / "tests").exists():
        PROJECT2_ROOT = candidate
        DAY1_ROOT = candidate.parent
        break
assert DAY1_ROOT is not None and PROJECT2_ROOT is not None, "找不到 day1/project2 项目根目录"
NOTEBOOK_ROOT = PROJECT2_ROOT / "notebooks"
for path in [str(DAY1_ROOT), str(PROJECT2_ROOT), str(NOTEBOOK_ROOT)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from notebook_utils import (
    check,
    check_equal,
    file_inventory,
    load_jsonl,
    masked_environment,
    run_command,
    run_unittest,
    show_markdown,
    show_table,
    source_excerpt,
)

RUN_LIVE_MODEL_TESTS = os.getenv("RUN_LIVE_MODEL_TESTS", "0") == "1"
print(f"Python: {sys.executable}")
print(f"DAY1_ROOT: {DAY1_ROOT}")
print(f"PROJECT2_ROOT: {PROJECT2_ROOT}")
print(f"RUN_LIVE_MODEL_TESTS: {RUN_LIVE_MODEL_TESTS}")

Python: D:\new things\项目1\day1\.venv\Scripts\python.exe
DAY1_ROOT: D:\new things\项目1\day1
PROJECT2_ROOT: D:\new things\项目1\day1\project2
RUN_LIVE_MODEL_TESTS: False


In [2]:
from pathlib import Path
import agent_graph
from handoff_repository import HandoffRepository
from memory_repository import MemoryRepository

temp_dir = tempfile.TemporaryDirectory()
root = Path(temp_dir.name)
handoff_repo = HandoffRepository(root / "handoff.sqlite3")
saver = agent_graph.create_sqlite_checkpointer(root / "checkpoints.sqlite3")
graph = agent_graph.build_graph(
    saver,
    handoff_repo,
    MemoryRepository(root / "memory.sqlite3"),
)
waiting = agent_graph.start_graph_agent(
    "小松PC200原厂液压泵要1件，多少钱？",
    thread_id="approval-thread",
    customer_id="customer-a",
    approval_mode="manual",
    graph=graph,
)
check_equal("报价前暂停审批", waiting["status"], "waiting_approval")
check("暂停前没有执行quote_tool", "quote_tool" not in waiting["called_tools"])
approved = agent_graph.resume_graph_agent(
    "approval-thread", "approve", comment="参数已核对", graph=graph
)
check_equal("批准后完成", approved["status"], "completed")
check("批准后执行quote_tool", "quote_tool" in approved["called_tools"])

D:\new things\项目1\day1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[PASS] 报价前暂停审批 | actual='waiting_approval', expected='waiting_approval'
[PASS] 暂停前没有执行quote_tool
[PASS] 批准后完成 | actual='completed', expected='completed'
[PASS] 批准后执行quote_tool


{'检查项': '批准后执行quote_tool', '状态': 'PASS', '说明': ''}

In [3]:
handed = agent_graph.start_graph_agent(
    "我要找人工客服确认PC200液压泵",
    thread_id="handoff-thread",
    customer_id="customer-a",
    approval_mode="auto",
    handoff_mode="manual",
    parser_mode="rules",
    graph=graph,
)
check_equal("明确要求人工后暂停", handed["status"], "waiting_human")
check("建立服务单", bool(handed["handoff_id"]))
resumed = agent_graph.resume_handoff_agent(
    "handoff-thread",
    "您好，我已经接手，请补充旧件号和铭牌照片。",
    agent_name="客服小王",
    graph=graph,
)
check_equal("人工回复后图执行完成", resumed["status"], "completed")
check_equal("记录接管客服", resumed["assigned_agent"], "客服小王")
check("人工回复写回原线程", "补充旧件号" in resumed["customer_reply"])
check(
    "执行轨迹包含人工回复节点",
    "human_response" in [item["step"] for item in resumed["execution_trace"]],
)
saver.conn.close()
temp_dir.cleanup()

[PASS] 明确要求人工后暂停 | actual='waiting_human', expected='waiting_human'
[PASS] 建立服务单
[PASS] 人工回复后图执行完成 | actual='completed', expected='completed'
[PASS] 记录接管客服 | actual='客服小王', expected='客服小王'
[PASS] 人工回复写回原线程
[PASS] 执行轨迹包含人工回复节点


In [4]:
handoff_tests = run_unittest(
    ["tests.test_handoff_runtime"],
    project2_root=PROJECT2_ROOT,
)
check("人工接管6条通过", "Ran 6 tests" in handoff_tests.output and "OK" in handoff_tests.output)

$ D:\new things\项目1\day1\.venv\Scripts\python.exe -m unittest tests.test_handoff_runtime -v
test_after_sales_approval_then_handoff_are_separate_interrupts (tests.test_handoff_runtime.HandoffRuntimeTests.test_after_sales_approval_then_handoff_are_separate_interrupts) ... ok
test_explicit_human_request_creates_case_and_resumes (tests.test_handoff_runtime.HandoffRuntimeTests.test_explicit_human_request_creates_case_and_resumes) ... ok
test_handoff_can_be_disabled_for_deterministic_baseline (tests.test_handoff_runtime.HandoffRuntimeTests.test_handoff_can_be_disabled_for_deterministic_baseline) ... ok
test_repeated_missing_information_routes_to_human (tests.test_handoff_runtime.HandoffRuntimeTests.test_repeated_missing_information_routes_to_human) ... ok
test_tool_error_routes_to_human_when_enabled (tests.test_handoff_runtime.HandoffRuntimeTests.test_tool_error_routes_to_human_when_enabled) ... ok
test_wechat_human_reply_enters_outbox_once (tests.test_handoff_runtime.HandoffRuntimeTests.tes

{'检查项': '人工接管6条通过', '状态': 'PASS', '说明': ''}

## 两类Human-in-the-loop不能混为一谈

- **工具审批interrupt：** 报价、售后等高风险动作执行前，批准、编辑后批准或拒绝。
- **人工客服interrupt：** 客户明确要求人工、重复缺信息、RAG证据不足、工具失败或高风险问题，创建服务单并等待真实回复。

### 面试官会问

1. 哪些工具需要人工审批，为什么？
2. 审批拒绝后如何保证工具没有执行？
3. 人工客服能看到哪些上下文？
4. 微信等非网页渠道为什么需要outbox和幂等？
5. 人工接管率、解决率和处理时长如何评估？

### 参考答案

1. **哪些工具需要审批？** 当前手动模式主要审批报价和售后工单：报价涉及价格承诺，售后涉及退款/换货等高风险动作。只读库存、物流估算和RAG通常不逐次审批，但异常或低置信结果仍可转人工。
2. **拒绝后怎样保证工具未执行？** interrupt位于dispatcher之前。拒绝恢复后，图把工具写入`skipped_tools`并记录`skip_tool/human_approval`轨迹，然后路由到下一个安全步骤，不进入实际调用节点；运行时测试同时断言`called_tools`中不存在该工具。
3. **人工客服看到什么？** 服务单携带原问题、客户/线程标识、解析意图和槽位、工具参数与已有结果、错误、接管原因、优先级和建议回复，让客服无需让客户重复描述；API Key、图片二进制和不必要隐私不进入上下文包。
4. **为什么需要outbox和幂等？** 微信webhook不能为等待人工保持长连接。人工回复先作为待发送消息写入outbox，再由渠道适配器异步投递；稳定去重键保证重试或checkpoint恢复不会向客户发送两次。
5. **怎样评估人工接管？** 接管率=`handoff/总会话`，还要按原因分层；解决率看服务单是否resolved；处理时长从创建到人工回复/结单；同时观察重复转接、建议回复采用率和人工SLA，避免单纯追求低接管率。

**代码落点：** `handoff_policy.py`、`handoff_repository.py`、`handoff_metrics.py`、`agent_graph.py::resume_handoff_agent`。